In [ ]:
!pip install langchain_community langchainhub chromadb langchain langchain-openai

In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('openai')



In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(web_paths=["https://educosys.com/course/all-courses"])
docs=loader.load()
print(docs)

[Document(metadata={'source': 'https://educosys.com/course/all-courses', 'title': 'Educosys', 'description': 'Unlock Your Potential with Keerti Purswani. Join Anytime. Get Lifetime Access to All Past, Ongoing and Future Batches', 'language': 'en'}, page_content='Educosys')]


In [ ]:
import os
os.environ["USER_AGENT"] = "EducosysRAGProject/1.0"

from langchain_community.document_loaders import WebBaseLoader

url = "https://educosys.com/course/all-courses"

loader = WebBaseLoader(
    web_paths=[url],
    header_template={
        "User-Agent": "Mozilla/5.0"
    }
)

docs = loader.load()

print("Number of documents:", len(docs))

for doc in docs:
    print("Metadata:", doc.metadata)
    print("Content preview:", doc.page_content[:3000])
    print("Content length:", len(doc.page_content))

Number of documents: 1
Metadata: {'source': 'https://educosys.com/course/all-courses', 'title': 'Educosys', 'description': 'Unlock Your Potential with Keerti Purswani. Join Anytime. Get Lifetime Access to All Past, Ongoing and Future Batches', 'language': 'en'}
Content preview: Educosys
Content length: 8


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

print("Original documents:", len(docs))
print("Split chunks:", len(splits))

Original documents: 1
Split chunks: 1


In [8]:
print("Total characters:", len(docs[0].page_content))
print(docs[0].page_content[:3000])

Total characters: 8
Educosys


In [9]:
for i, chunk in enumerate(splits):
    print(f"Chunk {i+1}:")
    print(chunk.page_content[:500])
    print("-" * 50)

Chunk 1:
Educosys
--------------------------------------------------


In [10]:
!pip install -q beautifulsoup4 requests

In [11]:
import requests
from bs4 import BeautifulSoup

url = "https://educosys.com/course/all-courses"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

for tag in soup(["script", "style", "nav", "footer", "header"]):
    tag.decompose()

text = soup.get_text(separator=" ", strip=True)

print("Extracted characters:", len(text))
print(text[:3000])

Status code: 200
Extracted characters: 8
Educosys


In [12]:

import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text(separator=" ", strip=True)

print("Extracted characters:", len(text))
print(text[:2000])


Status code: 200
Extracted characters: 1871
All products | Books to Scrape - Sandbox Books to Scrape We love being scraped! Home All products Books Travel Mystery Historical Fiction Sequential Art Classics Philosophy Romance Womens Fiction Fiction Childrens Religion Nonfiction Music Default Science Fiction Sports and Games Add a comment Fantasy New Adult Young Adult Science Poetry Paranormal Art Psychology Autobiography Parenting Adult Fiction Humor Horror History Food and Drink Christian Fiction Business Biography Thriller Contemporary Spirituality Academic Self Help Historical Christian Suspense Short Stories Novels Health Politics Cultural Erotica Crime All products 1000 results - showing 1 to 20 . Warning! This is a demo website for web scraping purposes. Prices and ratings here were randomly assigned and have no real meaning. A Light in the ... Â£51.77 In stock Add to basket Tipping the Velvet Â£53.74 In stock Add to basket Soumission Â£50.10 In stock Add to basket Sharp Objects Â

In [13]:

import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

soup = BeautifulSoup(response.text, "html.parser")

books = []

for item in soup.select("article.product_pod"):
    title = item.h3.a["title"]
    price = item.select_one(".price_color").get_text(strip=True)
    availability = item.select_one(".availability").get_text(strip=True)
    rating = item.select_one(".star-rating")["class"][1]

    books.append({
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability
    })

print("Books extracted:", len(books))
for book in books[:5]:
    print(book)


Books extracted: 20
{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three', 'availability': 'In stock'}
{'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One', 'availability': 'In stock'}
{'title': 'Soumission', 'price': 'Â£50.10', 'rating': 'One', 'availability': 'In stock'}
{'title': 'Sharp Objects', 'price': 'Â£47.82', 'rating': 'Four', 'availability': 'In stock'}
{'title': 'Sapiens: A Brief History of Humankind', 'price': 'Â£54.23', 'rating': 'Five', 'availability': 'In stock'}


In [14]:

from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            f"Book title: {book['title']}. "
            f"Price: {book['price']}. "
            f"Rating: {book['rating']}. "
            f"Availability: {book['availability']}."
        ),
        metadata={"title": book["title"]}
    )
    for book in books
]

print("Documents created:", len(docs))
print(docs[0].page_content)


Documents created: 20
Book title: A Light in the Attic. Price: Â£51.77. Rating: Three. Availability: In stock.


In [15]:

!pip install -q langchain-chroma langchain-huggingface sentence-transformers


In [16]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Convert text into numerical embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store the book documents in ChromaDB
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="coursebuddy_books"
)

print("Successfully stored", len(docs), "documents in ChromaDB!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully stored 20 documents in ChromaDB!


In [17]:

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

query = "Which books are available and what are their prices?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)



Result 1:
Book title: Set Me Free. Price: Â£17.46. Rating: Five. Availability: In stock.

Result 2:
Book title: Soumission. Price: Â£50.10. Rating: One. Availability: In stock.

Result 3:
Book title: Rip it Up and Start Again. Price: Â£35.02. Rating: Five. Availability: In stock.


In [18]:

!pip install -q langchain-openai


In [20]:

!pip install -q transformers accelerate


In [21]:

!pip install -q langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00


In [24]:

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get("groq"),
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are BookBuddy AI, a helpful book assistant.
Use only the retrieved information to answer.
If the information is insufficient, say so.

Context:
{context}

Question:
{question}

Answer:
""")

def ask_bookbuddy(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n".join(
        doc.page_content for doc in retrieved_docs
    )
    messages = prompt.format_messages(
        context=context,
        question=question
    )
    return llm.invoke(messages).content

print(ask_bookbuddy(
    "List the retrieved books with their prices and ratings."
))


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [25]:

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("groq"))

models = client.models.list()

for model in models.data:
    print(model.id)


canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
whisper-large-v3
allam-2-7b


In [27]:

from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("groq"),
    temperature=0
)

print("LLM initialized successfully!")


LLM initialized successfully!


In [28]:
print(ask_bookbuddy(
    "List the retrieved books with their prices and ratings."
))

- **Soumission** – Price: £50.10, Rating: One  
- **Rip it Up and Start Again** – Price: £35.02, Rating: Five  
- **Olio** – Price: £23.88, Rating: One
